In [ ]:
import torch
import yaml
from helper import load_default_dataloaders, load_ObjDet_model, load_best_model, get_bbox_preds, visualize_bbox_preds

In [ ]:
torch.manual_seed(42)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

## Loading Object Detection Model and Dataset

In [ ]:
dataset_path = "../kaggle/Traffic_Sign/car"
batch_size = 16
num_workers = 0 # multiple workers will not work in a ipynb on macOS

In [ ]:
# load class names from yaml file

with open(f"{dataset_path}/data.yaml", "r") as f:
    class_names = yaml.safe_load(f)["names"]

In [ ]:
# building the model with the new object detection head, getting the test dataloader from the weights preprocessing and the new input image size after the preprocessing transformations

model, test_transform, image_size = load_ObjDet_model(num_classes=len(class_names), frozen=True, device=device)

In [ ]:
_, _, test_dataloader = load_default_dataloaders(
    batch_size=batch_size,
    num_workers=num_workers, 
    image_size=image_size, 
    test_transform=test_transform, 
    dataset_path=dataset_path
)

In [ ]:
load_best_model(model)

In [ ]:
# compare the predicted bounding boxes with the GT bounding boxes
val_batch = next(iter(test_dataloader))
X, y = torch.stack(val_batch[0]).to(device), val_batch[1] # torch.stack() because the batch is a tuple of 64 images and 64 labels
pred = model(X)
pred_classes, pred_bboxes, pred_confidences = get_bbox_preds(batch_size=X.shape[0], pred=pred, threshold=0.25, image_size=image_size, nms_treshold=0.5)
visualize_bbox_preds(X.to('cpu'), pred_classes, pred_bboxes, pred_confidences, y=y, image_size=X.shape[-1], test_transforms=test_transform, class_names=class_names)